In [3]:
# import libraries
import cv2 
import json
import os

In [ ]:
#Set the linkpath
train_dir = "./train" # Add your link path to train file
annot_file = os.path.json(train_dir, "_annotations.coco.json") #file label _annotations.coco.json

out_occ = "processed/occupied"
out_emp = "processed/empty"

os.makedirs(out_occ, exist_ok = True)
os.makedirs(out_emp, exist_ok = True)

In [ ]:
target_size = (28,28)

with open (annot_file, "r") as f:
    coco = json.load(f)

image_map = {img["id"]: img["file_name"] for img in coco["images"]} #image id -> file name
category_map = {cat["id"]: cat["name"] for cat in coco["categories"]} # category id -> name
print("Categories:", category_map)

for ann in coco["annotations"]:
    image_id = ann["image_id"]
    category_id = ann["category_id"]
    bbox = ann["bbox"]  # [x, y, w, h]

    img_name = image_map[image_id]
    label_name = category_map[category_id].lower()

    img_path = os.path.join(TRAIN_DIR, img_name)
    image = cv2.imread(img_path)

    if image is None:
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_GBR2GRAY)
    x, y, w, h = map(int, bbox)

    # 1️⃣ Crop
    crop = gray[y:y+h, x:x+w]
    if crop.size == 0:
        continue

    # 2️⃣ Resize
    crop_28 = cv2.resize(
        crop, TARGET_SIZE, interpolation=cv2.INTER_AREA
    )

    # 3️⃣ Chọn thư mục theo label
    if "occupied" in label_name or "car" in label_name:
        save_dir = out_occ
    else:
        save_dir = out_emp

    save_name = f"{img_name[:-4]}_{ann['id']}.png"
    cv2.imwrite(os.path.join(save_dir, save_name), crop_28)

print("✔ HOÀN TẤT: COCO → 28x28 Grayscale Dataset")